In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import pandas as pd
from datetime import datetime
import pytz
import time
from io import BytesIO
import zipfile
from google.cloud import storage, bigquery
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
from google.api_core.exceptions import NotFound, GoogleAPICallError

In [3]:
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
RUTA = 'TRAMAS DE BAJAS'
DATASET_ID = "produccion"
TABLE_CONCILIACION_ID= "BASES_CONCILIACION"

FECHA_INICIO= '2026-03-27'
FECHA_FIN= '2026-04-30'

In [ ]:
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)
zh_peru = pytz.timezone("America/Lima")

def tabla_existe(table_ref):
    try:
      tabla = bigquery_client.get_table(table_ref)
      return True
    except NotFound:
        # La tabla no existe
        return False
    except GoogleAPICallError as e:
        # Otros errores de BigQuery (permisos, conexión, etc.)
        print(f"⚠️ Error al consultar BigQuery: {e}")
        raise

def Consultar_bajas(fecha_inicio, fecha_fin):
    # Verificar si existe la tabla de control
    table_conciliaciones_ref = bigquery_client.dataset(DATASET_ID).table(TABLE_CONCILIACION_ID)
    if not tabla_existe(table_conciliaciones_ref):
        return print(f"⚠️ Error al consultar BigQuery")
    else:
        # Buscar archivo en la tabla de control
        query = f"""
            WITH bajas_tramas AS (
              SELECT
                  CASE
                        WHEN TIPO_DE_SEGURO IN ('901','902','903','906','941','951',
                                              '952','953','954','958','959','961',
                                              '962','963','964','966','968','972') THEN 'PRESTAMOS'
                        ELSE TIPO_DE_SEGURO
                  END AS COD_PRODUCTO, TIPO_DE_MOVIMIENTO, TRAMA_ORIGINAL,
                  CONCAT(SUBSTR(CERTIFICADO, 1,4),SUBSTR(CERTIFICADO, 7,2), SUBSTR(CERTIFICADO, 11,10)) AS CERT_CRUCE
              FROM `produccion.TRAMAS_diarias_BBVA_2026`
              WHERE TIPO_DE_MOVIMIENTO IN ('2','6') AND
              FECHA_TRAMA BETWEEN @fecha_desde AND @fecha_hasta
            ),
            bases AS (
              SELECT CERTIFICADO_BANCO, FECHA_OPERACION, PRIMA,
                    CONCAT(SUBSTR(CERTIFICADO_BANCO, 1,4),SUBSTR(CERTIFICADO_BANCO, 7,2), SUBSTR(CERTIFICADO_BANCO, 11,10)) AS CERT_CRUCE_BASE
              FROM `{table_conciliaciones_ref}`
            )
            SELECT DISTINCT COD_PRODUCTO, TIPO_DE_MOVIMIENTO, TRAMA_ORIGINAL
            FROM bajas_tramas T
            LEFT JOIN bases B ON T.CERT_CRUCE = B.CERT_CRUCE_BASE
            WHERE B.CERTIFICADO_BANCO IS NULL
        """
        job_config = bigquery.QueryJobConfig(query_parameters=
                                          [bigquery.ScalarQueryParameter("fecha_desde", "STRING", FECHA_INICIO),
                                            bigquery.ScalarQueryParameter("fecha_hasta", "STRING", FECHA_FIN)])

        df = bigquery_client.query(query, job_config=job_config).to_dataframe()
        return df
##############################################################################################
df_consulta= Consultar_bajas(FECHA_INICIO, FECHA_FIN)

# Fecha y hora actual
zh_peru = pytz.timezone("America/Lima")
fecha_hora = datetime.now(zh_peru).strftime("%Y-%m-%d_%H%M%S")
ruta_base = f"{RUTA}/{fecha_hora}"

bucket = storage_client.bucket(BUCKET_NAME)
"""
# Agrupar por producto
for tipo, grupo in df_consulta.groupby("COD_PRODUCTO"):
    nombre_archivo = tipo + ".txt"
    blob_path = f"{ruta_base}/{nombre_archivo}"

    blob = bucket.blob(blob_path)
    with blob.open("w", encoding="utf-8") as f:
        for trama in grupo["TRAMA_ORIGINAL"].dropna():
            if not trama.endswith("\n"):
                trama += "\n"
            f.write(trama)
    print(f"ARCHIVO CREADO EN: {blob_path}")
  """

  # Agrupar por producto
zip_buffer = BytesIO()

with zipfile.ZipFile(zip_buffer, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
    for tipo, grupo in df_consulta.groupby("COD_PRODUCTO"):
        nombre_archivo = tipo + ".txt"
        blob_path = f"{ruta_base}/{nombre_archivo}"

        contenido = []
        blob = bucket.blob(blob_path)
        with blob.open("w", encoding="utf-8") as f:
            for trama in grupo["TRAMA_ORIGINAL"].dropna():
                if not trama.endswith("\n"):
                    trama += "\n"
                f.write(trama)
                contenido.append(trama)
        print(f"ARCHIVO CREADO EN: {blob_path}")

        contenido_str = "".join(contenido)
        # Agregar al ZIP
        zf.writestr(nombre_archivo, contenido_str)

zip_buffer.seek(0)
# Subir el ZIP a GCS
zip_blob_path = f"{ruta_base}/archivos_comprimidos.zip"
blob_zip = bucket.blob(zip_blob_path)
blob_zip.upload_from_file(zip_buffer, content_type="application/zip")

print(f"ZIP creado en: {zip_blob_path}")

ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/703.txt
ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/800.txt
ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/801.txt
ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/803.txt
ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/803_TLMK.txt
ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/804.txt
ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/808.txt
ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/809.txt
ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/812.txt
ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/816.txt
ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/818.txt
ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/822.txt
ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/825.txt
ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/827.txt
ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/829.txt
ARCHIVO CREADO EN: TRAMAS DE BAJAS/2026-05-20_175106/904.txt
ARCHIVO CREADO EN: 